# 03 — Results: las figuras finales

**Va en:** `fraud-review-queue/notebooks/03_results.ipynb`

Fábrica de figuras (delegable, plan §5.3). El **write-up es tuyo** — este
notebook solo produce los PNG de `reports/figures/` y las tablas en markdown
para pegar en el README. Prosa en español (notebook de trabajo); **todo lo que
sale hacia el README — etiquetas, títulos de figuras, tablas — va en inglés.**

## Insumos (todos medidos, ninguno inventado)

| Archivo | Lo produce |
|---|---|
| `reports/scored_test.parquet` | La única mirada del Día 6 |
| `reports/policy_comparison.csv` | Ídem |
| `reports/scored_calib.parquet` (`score_raw`, `p`, `isFraud`) | Cierre del Día 5 |

Cada celda **falla ruidosamente o avisa** si falta su insumo. La regla §11 de
instrucciones: jamás publicar un número que no se midió — un hueco visible es
mejor que un placeholder plausible.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- insumos y salidas -------------------------------------------------------
REPORTS = Path("../reports")
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

SCORED_TEST = REPORTS / "scored_test.parquet"
POLICY_CSV = REPORTS / "policy_comparison.csv"
SCORED_CALIB = REPORTS / "scored_calib.parquet"

# --- tu config (fuente única §9.1) -------------------------------------------
from fraudq.config import COSTS, POLICY, SENSITIVITY_RANGES  # ajusta nombres si difieren

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / name, bbox_inches="tight")
    print(f"-> {FIGURES / name}")

## Figura 1 — La protagonista: qué cuesta cada política

Costo por \$1,000 de las cuatro políticas, con el ahorro (3)→(4) anotado.
**Es la figura de arriba del README** (§14.2) — y su número es la primera
línea (§14.1).

In [ ]:
assert POLICY_CSV.exists(), "FALTA policy_comparison.csv — protocolo del Día 6"
comparison = pd.read_csv(POLICY_CSV, index_col=0)

labels = {
    "approve_all": "Approve\neverything",
    "single_threshold": "Single score\nthreshold",
    "topk_by_score": "Review top-K\nby score",
    "topk_by_value": "Review top-K\nby value (ours)",
}
vals = comparison["cost_per_1k"]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars = ax.bar([labels[i] for i in comparison.index], vals,
              color=["#adb5bd", "#adb5bd", "#e76f51", "#2a9d8f"])
ax.bar_label(bars, fmt="$%.2f", padding=3)
ax.set_ylabel("Expected loss per $1,000 of volume")
ax.set_title("Ranking the review queue by value, not score")

sav = vals["topk_by_score"] - vals["topk_by_value"]
ax.annotate(f"${sav:.2f} per $1,000\nleft on the table",
            xy=(3, vals["topk_by_value"]), xytext=(2.05, vals["topk_by_score"] * 1.05),
            arrowprops=dict(arrowstyle="->"), fontsize=10, fontweight="bold")
save(fig, "fig1_policy_comparison.png")

print(comparison.to_markdown(floatfmt=".2f"))  # tabla §14.5, lista para el README

## Figura 2 — Calibración antes / después (§7.2)

La evidencia empírica de la regla contracultural: el score crudo no era una
probabilidad; el calibrador sí. Insumo: `scored_calib.parquet` del Día 5.

In [ ]:
if not SCORED_CALIB.exists():
    print("FALTA scored_calib.parquet (persistelo del Día 5) — figura pendiente")
else:
    from fraudq.evaluate.metrics import reliability_table, brier_score, ece

    calib = pd.read_parquet(SCORED_CALIB)
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True)
    for ax, col, title in [(axes[0], "score_raw", "Before: raw GBDT score"),
                           (axes[1], "p", "After: calibrated probability")]:
        t = reliability_table(calib["isFraud"], calib[col], n_bins=10)
        ax.plot([0, 1], [0, 1], "--", lw=1, color="gray")
        ax.plot(t["mean_p"], t["frac_pos"], "o-")
        b = brier_score(calib["isFraud"], calib[col])
        e = ece(calib["isFraud"], calib[col])
        ax.set_title(f"{title}\nBrier {b:.4f} · ECE {e:.4f}", fontsize=10)
        ax.set_xlabel("Predicted probability")
    axes[0].set_ylabel("Observed fraud rate")
    save(fig, "fig2_calibration_before_after.png")

## Figura 3 — El tornado (§8.3)

Usa TU `tornado_data` del Día 7 sobre el scoring persistido. Si aún no lo
implementaste, la celda te lo dice en vez de inventar barras.

In [ ]:
assert SCORED_TEST.exists(), "FALTA scored_test.parquet — protocolo del Día 6"
scored = pd.read_parquet(SCORED_TEST)

from fraudq.evaluate.policies import compare_policies, fit_single_threshold
from fraudq.evaluate.sensitivity import tornado_data, plot_tornado, savings_per_1k

THRESHOLD = fit_single_threshold(scored, COSTS)  # o el t que fijaste en calib

def evaluate(cfg):
    return compare_policies(scored, cfg, POLICY.daily_capacity_pct, THRESHOLD)

try:
    tornado = tornado_data(COSTS, SENSITIVITY_RANGES, evaluate)
    base = savings_per_1k(evaluate(COSTS))
    ax = plot_tornado(tornado, base_savings=base)
    save(ax.figure, "fig3_tornado.png")
    print(tornado.to_markdown(index=False, floatfmt=".2f"))
except NotImplementedError as e:
    print(f"PENDIENTE: {e} — completa sensitivity.tornado_data (Día 7)")

## Figura 4 — Regiones de decisión + la distribución real (p, monto)

Panel izquierdo: las fronteras del §2.4 (dependen del monto — la política de
un umbral es estructuralmente equivocada). Panel derecho: dónde vive de verdad
la masa de (p, monto) en test — **la figura de la contingencia §13.5**: si el
efecto salió chico, este panel es la razón empírica y pasa a ser protagonista.

In [ ]:
from fraudq.policy.costs import cost_approve, cost_block, value_of_review

p_grid = np.linspace(0.001, 0.999, 400)
a_grid = np.geomspace(5, 2000, 400)
P, A = np.meshgrid(p_grid, a_grid)
region = np.where(value_of_review(P, A, COSTS) > 0, 1,
                  np.where(cost_approve(P, A, COSTS) <= cost_block(P, A, COSTS), 0, 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
axes[0].contourf(P, A, region, levels=[-0.5, 0.5, 1.5, 2.5],
                 colors=["#2a9d8f", "#e9c46a", "#e76f51"], alpha=0.4)
axes[0].set_yscale("log")
axes[0].set_title("Decision regions (base cost assumptions)", fontsize=10)
axes[0].set_ylabel("Amount ($, log)")

h = axes[1].hexbin(scored["p"].clip(1e-3, 1 - 1e-3), scored["TransactionAmt"].clip(5, 2000),
                   yscale="log", gridsize=45, bins="log", cmap="viridis")
axes[1].set_title("Where the test data actually lives", fontsize=10)
fig.colorbar(h, ax=axes[1], label="log10(count)")
for ax in axes:
    ax.set_xlabel("Calibrated fraud probability")
save(fig, "fig4_regions_and_joint_distribution.png")

## Figura 5 — Las dos colas son (casi) disjuntas (§2.6)

El solape entre el top-K por score y el top-K por valor, día a día. Si el
solape es bajo, la frase "los dos conjuntos son casi disjuntos" queda medida,
no afirmada.

In [ ]:
from types import SimpleNamespace
from fraudq.evaluate.policies import actions_topk_by_score, actions_topk_by_value
from fraudq.policy.simulate import simulate_queue

qs = simulate_queue(scored, actions_topk_by_score, COSTS, POLICY.daily_capacity_pct)
qv = simulate_queue(scored, actions_topk_by_value, COSTS, POLICY.daily_capacity_pct)
s_set = set(np.flatnonzero((qs.actions == "review").to_numpy()))
v_set = set(np.flatnonzero((qv.actions == "review").to_numpy()))
overlap = len(s_set & v_set) / max(len(s_set | v_set), 1)
print(f"Jaccard overlap of the two queues: {overlap:.1%}")

both, only_s, only_v = s_set & v_set, s_set - v_set, v_set - s_set
fig, ax = plt.subplots(figsize=(7, 4.4))
for idx, label, color in [(only_s, "Score queue only", "#e76f51"),
                          (only_v, "Value queue only", "#2a9d8f"),
                          (both, "Both", "#666666")]:
    sub = scored.iloc[sorted(idx)]
    ax.scatter(sub["p"], sub["TransactionAmt"], s=6, alpha=0.35, label=label, color=color)
ax.set_yscale("log"); ax.legend(markerscale=3)
ax.set_xlabel("Calibrated fraud probability"); ax.set_ylabel("Amount ($, log)")
ax.set_title(f"Two review queues, {overlap:.0%} overlap", fontsize=10)
save(fig, "fig5_queue_overlap.png")

## Figura 6 — Drift y degradación (§8.4)

La frase de cadencia de reentrenamiento del README sale de aquí.

In [ ]:
from fraudq.evaluate.drift import performance_by_month

perf = performance_by_month(scored)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(perf["month"], perf["pr_auc"], "o-")
ax.set_xlabel("Relative month of the test window"); ax.set_ylabel("PR-AUC")
ax.set_title("Performance decay over the test period", fontsize=10)
save(fig, "fig6_performance_by_month.png")
print(perf.to_markdown(index=False, floatfmt=".4f"))
decay = (perf["pr_auc"].iloc[0] - perf["pr_auc"].iloc[-1]) / perf["pr_auc"].iloc[0]
print(f"Decay first->last month: {decay:.1%}  (la frase del README sale de aquí)")

## Cierre

Con las figuras en `reports/figures/`, lo que queda es **tuyo** (§5.3): el
README final y el write-up. La checklist §14/§15 está en `README-dia-9.md`.
Regla final: cada número del README debe poder rastrearse a una celda de este
notebook o a un archivo de `reports/` — **ninguno se escribe de memoria.**